In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
print(df.shape)
print(df.info())
df.head()

(7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Check if there are any non-numeric values causing this
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# After conversion, check how many became NaN (these were the problematic rows)
print(df['TotalCharges'].isnull().sum())

11


In [3]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# confirm no more missing values
print(df['TotalCharges'].isnull().sum())
print(df.shape)

0
(7043, 21)


In [4]:
# Tenure buckets
def tenure_bucket(t):
    if t <= 12:
        return "0-1yr"
    elif t <= 24:
        return "1-2yr"
    elif t <= 48:
        return "2-4yr"
    else:
        return "4yr+"

df['tenure_group'] = df['tenure'].apply(tenure_bucket)

In [5]:
#Total number of services subscribed (engagement proxy — higher services -> higher the customer  "locked-in" (hypothesis):
services = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

def count_services(row):
    return sum(1 for col in services if row[col] == 'Yes')

df['total_services'] = df.apply(count_services, axis=1)

In [6]:
#Average monthly spend per tenure month 
df['avg_monthly_spend'] = df['TotalCharges'] / df['tenure'].replace(0, 1)  # avoid divide by zero

In [7]:
# Step A: Binary columns - simple mapping
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# Target variable encoding
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(df[binary_cols + ['gender', 'Churn']].head())

   Partner  Dependents  PhoneService  PaperlessBilling  gender  Churn
0        1           0             0                 1       0      0
1        0           0             1                 0       1      0
2        0           0             1                 1       1      1
3        0           0             0                 0       1      0
4        0           0             1                 1       0      1


In [8]:
#droping customerid column bcz it does not represent any customer behaviour as it is a unique identifier /label
df = df.drop('customerID', axis=1)
print(df.shape)

(7043, 23)


In [9]:
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                   'Contract', 'PaymentMethod', 'tenure_group']

df_encoded = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

print(df_encoded.shape)
df_encoded.head()

(7043, 36)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_1-2yr,tenure_group_2-4yr,tenure_group_4yr+
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,True,False,False,False,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,True,False,False,False,True,False,True,False
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,True,False,False,False
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,False,False,True,False,False,False,False,False,True,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,True,False,False,False,False


In [10]:
df_encoded.to_csv('../data/processed/churn_cleaned_encoded.csv', index=False)
print("Saved processed data!")

Saved processed data!
